# Acquisition Simulation: Predictive Utility of Elution Sequence Models

**Version: 1.2** | 2026-03-09

**Purpose:** Evaluate whether elution sequence predictions can improve LC-MS/MS
acquisition by anticipating which analytes will elute in the near future.

**Question:** If the instrument could "look ahead" using model predictions, how
often would the predicted m/z targets actually appear in the next 5, 15, or 30 seconds?

**Simulation approach:**
- At each position in a test sample's elution sequence, the model predicts the next m/z bin
- We check whether that prediction (or top-k predictions) match any feature appearing
  within the specified time horizon
- This simulates a real-time acquisition controller that pre-schedules MS/MS scans
  for predicted targets

**Models:** LSTM (98.38% top-1) and Transformer (98.05% top-1) from notebook 01

**Runtime:** GPU recommended for faster inference; CPU works but slower.

**Changelog:**
- v1.2: Fix Transformer checkpoint loading — `strict=False` to skip `causal_mask` buffer
- v1.1: Fix `total_mem` → `total_memory` GPU property; fix `self.embed` → `self.embedding` to match checkpoint keys
- v1.0: Initial version


In [ ]:
# @title Cell 1: Setup and Google Drive mount
import os
import time

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = "/content/drive/MyDrive/0000 Fun with coding/088 Lights-Out R01 Grant/Specific Aim 1/poc3_elution_sequence"
EXPERIMENT_DIR = f"{DRIVE_DIR}/06_acquisition_simulation"
OUTPUT_DIR = f"{EXPERIMENT_DIR}/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Data and model paths
TRAIN_DATA = f"{DRIVE_DIR}/01_train_models/tokenized_features.parquet"
LSTM_CKPT = f"{DRIVE_DIR}/01_train_models/outputs/lstm_best.pt"
TFM_CKPT = f"{DRIVE_DIR}/01_train_models/outputs/transformer_best.pt"

print(f"Experiment dir: {EXPERIMENT_DIR}")
print(f"Output dir:     {OUTPUT_DIR}")
print(f"Training data:  {os.path.exists(TRAIN_DATA)}")
print(f"LSTM checkpoint: {os.path.exists(LSTM_CKPT)}")
print(f"Transformer checkpoint: {os.path.exists(TFM_CKPT)}")


In [ ]:
# @title Cell 2: Import dependencies
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from collections import defaultdict

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


In [ ]:
# @title Cell 3: Configuration
# Tokenization constants (must match training)
MZ_BIN_WIDTH = 10          # Da per m/z bin
RT_BIN_WIDTH = 3           # seconds per RT bin
CONTEXT_LENGTH = 64        # tokens of context
RANDOM_SEED = 42
BATCH_SIZE = 64            # larger batch for inference only

# Prediction horizons (seconds)
HORIZONS = [5, 15, 30]

# Top-k values to evaluate
TOP_K_VALUES = [1, 3, 5, 10]

# Split fractions (must match training)
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15

# Categorical encodings (must match training)
RT_GAP_LABELS = ["co-elute", "0.1-0.5s", "0.5-1s", "1-2s", "2-5s", "5-15s", ">15s"]
RT_GAP_BINS = [0, 0.1, 0.5, 1.0, 2.0, 5.0, 15.0, float("inf")]
INTENSITY_RANK_LABELS = ["top1%", "top5%", "top20%", "top50%", "low"]
INTENSITY_RANK_BINS = [0, 0.01, 0.05, 0.20, 0.50, 1.0]
POLARITY_MAP = {"pos": 0, "neg": 1, "unk": 2}
RT_GAP_MAP = {label: i for i, label in enumerate(RT_GAP_LABELS)}
INTENSITY_MAP = {label: i for i, label in enumerate(INTENSITY_RANK_LABELS)}

# Model hyperparameters
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 2
NUM_HEADS = 4
FF_DIM = 256
DROPOUT = 0.1

print("Configuration loaded")
print(f"Horizons: {HORIZONS} seconds")
print(f"Top-k values: {TOP_K_VALUES}")


## 1. Load Data and Rebuild Test Set

We reload the tokenized feature table and apply the same sample-aware split
used during training (seed=42) to recover the exact test set.


In [ ]:
# @title Cell 4: Load data and rebuild test split
tok = pd.read_parquet(TRAIN_DATA)
tok = tok.sort_values(["study", "sample_id", "seq_pos"])

# Encode categorical fields
tok["polarity_idx"] = tok["polarity_tok"].map(POLARITY_MAP).fillna(2).astype(int)
tok["rt_gap_idx"] = tok["rt_gap_bin"].astype(str).map(RT_GAP_MAP).fillna(0).astype(int)
tok["intensity_idx"] = tok["intensity_rank"].astype(str).map(INTENSITY_MAP).fillna(4).astype(int)

# Compute RT in seconds (rt is in minutes in the parquet)
tok["rt_seconds"] = tok["rt"] * 60

# Sample-aware split (same as training)
rng = np.random.RandomState(RANDOM_SEED)
samples = tok[["study", "sample_id"]].drop_duplicates().reset_index(drop=True)
train_keys, val_keys, test_keys = set(), set(), set()

for study, group in samples.groupby("study"):
    indices = group.index.values.copy()
    rng.shuffle(indices)
    n = len(indices)
    n_test = max(1, int(n * TEST_FRACTION))
    n_val = max(1, int(n * VAL_FRACTION))
    for idx in indices[:n_test]:
        test_keys.add((group.loc[idx, "study"], group.loc[idx, "sample_id"]))
    for idx in indices[n_test:n_test + n_val]:
        val_keys.add((group.loc[idx, "study"], group.loc[idx, "sample_id"]))
    for idx in indices[n_test + n_val:]:
        train_keys.add((group.loc[idx, "study"], group.loc[idx, "sample_id"]))

test_mask = tok.apply(lambda r: (r["study"], r["sample_id"]) in test_keys, axis=1)
test_df = tok[test_mask].copy()

# Also get full data for MAX_MZ_BIN computation
MAX_MZ_BIN = int(tok["mz_bin"].max()) + 1

print(f"Total features: {len(tok):,}")
print(f"Test samples: {len(test_keys)}")
print(f"Test features: {len(test_df):,}")
print(f"MAX_MZ_BIN (num classes): {MAX_MZ_BIN}")
print(f"RT range: {test_df['rt_seconds'].min():.0f}s - {test_df['rt_seconds'].max():.0f}s")

# Build per-sample arrays with RT in seconds
test_samples = []
for (study, sid), group in test_df.groupby(["study", "sample_id"]):
    g = group.sort_values("seq_pos")
    test_samples.append({
        "study": study,
        "sample_id": sid,
        "mz_bin": g["mz_bin"].values.astype(np.int64),
        "md_bin": g["md_bin"].values.astype(np.int64),
        "rt_gap_idx": g["rt_gap_idx"].values.astype(np.int64),
        "polarity_idx": g["polarity_idx"].values.astype(np.int64),
        "intensity_idx": g["intensity_idx"].values.astype(np.int64),
        "rt_bin": g["rt_bin"].values.astype(np.int64),
        "rt_seconds": g["rt_seconds"].values,  # actual RT in seconds
    })

print(f"\nTest samples built: {len(test_samples)}")
for s in test_samples[:3]:
    print(f"  {s['study']}/{s['sample_id']}: {len(s['mz_bin'])} features, "
          f"RT {s['rt_seconds'][0]:.0f}-{s['rt_seconds'][-1]:.0f}s")


## 2. Model Definitions

Same architectures as notebook 01 (duplicated for self-containment).


In [ ]:
# @title Cell 5: Model definitions (LSTM + Transformer)
class MultiFieldEmbedding(nn.Module):
    def __init__(self, embedding_dim, max_mz_bin=120, max_md_bin=20,
                 max_rt_gap=7, max_polarity=3, max_intensity=5):
        super().__init__()
        self.mz_embed = nn.Embedding(max_mz_bin, embedding_dim)
        self.md_embed = nn.Embedding(max_md_bin, embedding_dim)
        self.gap_embed = nn.Embedding(max_rt_gap, embedding_dim)
        self.pol_embed = nn.Embedding(max_polarity, embedding_dim)
        self.int_embed = nn.Embedding(max_intensity, embedding_dim)

    def forward(self, mz, md, gap, pol, intensity):
        return (self.mz_embed(mz) + self.md_embed(md) + self.gap_embed(gap)
                + self.pol_embed(pol) + self.int_embed(intensity))


class LSTMModel(nn.Module):
    def __init__(self, num_mz_classes, embedding_dim=64, hidden_dim=128,
                 num_layers=2, dropout=0.1, **embed_kwargs):
        super().__init__()
        self.embedding = MultiFieldEmbedding(embedding_dim, **embed_kwargs)
        self.lstm = nn.LSTM(embedding_dim, hidden_dim, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0)
        self.head = nn.Linear(hidden_dim, num_mz_classes)
        self.dropout = nn.Dropout(dropout)

    def forward(self, mz, md, gap, pol, intensity):
        x = self.embedding(mz, md, gap, pol, intensity)
        out, _ = self.lstm(x)
        return self.head(self.dropout(out[:, -1, :]))


class TransformerModel(nn.Module):
    def __init__(self, num_mz_classes, embedding_dim=64, num_heads=4,
                 ff_dim=256, num_layers=2, dropout=0.1,
                 context_length=64, **embed_kwargs):
        super().__init__()
        self.embedding = MultiFieldEmbedding(embedding_dim, **embed_kwargs)
        self.pos_embed = nn.Embedding(context_length, embedding_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim, nhead=num_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        self.head = nn.Linear(embedding_dim, num_mz_classes)
        self.dropout = nn.Dropout(dropout)
        self.context_length = context_length

    def forward(self, mz, md, gap, pol, intensity):
        B, T = mz.shape
        x = self.embedding(mz, md, gap, pol, intensity)
        pos = torch.arange(T, device=mz.device).unsqueeze(0).expand(B, T)
        x = x + self.pos_embed(pos)
        mask = torch.triu(torch.ones(T, T, device=mz.device), diagonal=1).bool()
        out = self.transformer(x, mask=mask)
        return self.head(self.dropout(out[:, -1, :]))


def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model classes defined")


In [ ]:
# @title Cell 6: Load trained models from checkpoints
embed_kw = dict(max_mz_bin=MAX_MZ_BIN, max_md_bin=20, max_rt_gap=7,
                max_polarity=3, max_intensity=5)

# --- Load LSTM ---
lstm_ckpt = torch.load(LSTM_CKPT, map_location=device, weights_only=False)
lstm_config = lstm_ckpt["config"]
lstm_model = LSTMModel(
    lstm_config["num_classes"], lstm_config["embedding_dim"],
    lstm_config["hidden_dim"], lstm_config["num_layers"], DROPOUT, **embed_kw
)
lstm_model.load_state_dict(lstm_ckpt["model_state_dict"])
lstm_model = lstm_model.to(device)
lstm_model.eval()
print(f"LSTM loaded: {count_parameters(lstm_model):,} params, "
      f"best epoch {lstm_ckpt['best_epoch']}")

# --- Load Transformer ---
tfm_ckpt = torch.load(TFM_CKPT, map_location=device, weights_only=False)
tfm_config = tfm_ckpt["config"]
tfm_model = TransformerModel(
    tfm_config["num_classes"], tfm_config["embedding_dim"],
    tfm_config["num_heads"], tfm_config["ff_dim"],
    tfm_config["num_layers"], DROPOUT,
    tfm_config["context_length"], **embed_kw
)
tfm_model.load_state_dict(tfm_ckpt["model_state_dict"], strict=False)
tfm_model = tfm_model.to(device)
tfm_model.eval()
print(f"Transformer loaded: {count_parameters(tfm_model):,} params, "
      f"best epoch {tfm_ckpt['best_epoch']}")


## 3. Acquisition Simulation

**How it works:**

For each position `t` in a test sample (after the initial context window):
1. Feed the preceding 64 tokens to the model
2. Get top-k predicted m/z bins for the next eluting feature
3. Look at all features that actually elute within the next H seconds (horizon)
4. **Hit** = at least one of the top-k predictions matches an actual future feature's m/z bin

**Metrics:**
- **Hit rate** — fraction of predictions where at least one top-k target appears in the horizon
- **Coverage** — fraction of actual future features that were predicted
- **Unique targets per horizon** — how many distinct m/z bins appear in each time window

This simulates what an instrument controller could do: at each moment, use the model
to generate a "watch list" of m/z values to pre-schedule MS/MS scans for.


In [ ]:
# @title Cell 7: Run acquisition simulation
def run_acquisition_sim(model, test_samples, horizons, top_k_values, device):
    """Simulate predictive acquisition for each test sample.

    For each prediction position:
    - Get model's top-k predictions
    - Check which features actually appear within each time horizon
    - Record hit/miss for each (horizon, k) combination
    """
    model.eval()
    results = {h: {k: {"hits": 0, "total": 0, "coverage_num": 0, "coverage_den": 0}
                   for k in top_k_values}
               for h in horizons}

    # Track per-position data for detailed analysis
    position_data = []

    with torch.no_grad():
        for si, sample in enumerate(test_samples):
            n_features = len(sample["mz_bin"])
            rt_sec = sample["rt_seconds"]

            for pos in range(CONTEXT_LENGTH, n_features):
                # Build context
                start = pos - CONTEXT_LENGTH
                ctx_mz = torch.tensor(sample["mz_bin"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
                ctx_md = torch.tensor(sample["md_bin"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
                ctx_gap = torch.tensor(sample["rt_gap_idx"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
                ctx_pol = torch.tensor(sample["polarity_idx"][start:pos], dtype=torch.long).unsqueeze(0).to(device)
                ctx_int = torch.tensor(sample["intensity_idx"][start:pos], dtype=torch.long).unsqueeze(0).to(device)

                # Get predictions
                logits = model(ctx_mz, ctx_md, ctx_gap, ctx_pol, ctx_int)  # (1, num_classes)

                current_rt = rt_sec[pos - 1]  # RT of the last observed feature

                for h in horizons:
                    # Find all features within horizon
                    future_mask = (rt_sec[pos:] > current_rt) & (rt_sec[pos:] <= current_rt + h)
                    future_mz_bins = set(sample["mz_bin"][pos:][future_mask])

                    if len(future_mz_bins) == 0:
                        continue  # no features in this horizon (e.g., end of run)

                    for k in top_k_values:
                        _, topk_preds = logits.topk(k, dim=1)
                        pred_set = set(topk_preds[0].cpu().numpy())

                        # Hit: at least one prediction matches a future feature
                        hit = len(pred_set & future_mz_bins) > 0
                        results[h][k]["hits"] += int(hit)
                        results[h][k]["total"] += 1

                        # Coverage: fraction of future features that were predicted
                        covered = len(pred_set & future_mz_bins)
                        results[h][k]["coverage_num"] += covered
                        results[h][k]["coverage_den"] += len(future_mz_bins)

                # Record position data (for top-1 at 5s only, to avoid memory bloat)
                _, top1_pred = logits.topk(1, dim=1)
                actual_next_mz = sample["mz_bin"][pos] if pos < n_features else -1
                rt_frac = current_rt / rt_sec[-1] if rt_sec[-1] > 0 else 0
                position_data.append({
                    "sample_idx": si,
                    "pos": pos,
                    "rt_seconds": current_rt,
                    "rt_fraction": rt_frac,
                    "pred_mz": int(top1_pred[0, 0].cpu()),
                    "actual_mz": int(actual_next_mz),
                    "correct": int(top1_pred[0, 0].cpu()) == int(actual_next_mz),
                })

            if (si + 1) % 10 == 0:
                print(f"  Processed {si + 1}/{len(test_samples)} samples")

    return results, pd.DataFrame(position_data)

# Run for both models
print("=" * 60)
print("Running acquisition simulation — LSTM")
print("=" * 60)
t0 = time.time()
lstm_results, lstm_pos_data = run_acquisition_sim(
    lstm_model, test_samples, HORIZONS, TOP_K_VALUES, device)
print(f"LSTM done in {time.time()-t0:.0f}s")

print()
print("=" * 60)
print("Running acquisition simulation — Transformer")
print("=" * 60)
t0 = time.time()
tfm_results, tfm_pos_data = run_acquisition_sim(
    tfm_model, test_samples, HORIZONS, TOP_K_VALUES, device)
print(f"Transformer done in {time.time()-t0:.0f}s")


## 4. Results


In [ ]:
# @title Cell 8: Display results table
def print_results(name, results):
    print(f"\n{'='*70}")
    print(f"ACQUISITION SIMULATION RESULTS — {name}")
    print(f"{'='*70}")
    print(f"{'Horizon':>10}  {'Top-k':>6}  {'Hit Rate':>10}  {'Coverage':>10}  {'n':>8}")
    print("-" * 50)
    for h in HORIZONS:
        for k in TOP_K_VALUES:
            r = results[h][k]
            if r["total"] > 0:
                hit_rate = r["hits"] / r["total"]
                coverage = r["coverage_num"] / r["coverage_den"] if r["coverage_den"] > 0 else 0
                print(f"{h:>8}s  {k:>6}  {hit_rate:>10.1%}  {coverage:>10.1%}  {r['total']:>8,}")

print_results("LSTM", lstm_results)
print_results("Transformer", tfm_results)


## 5. Random Baseline Comparison

To contextualize the hit rates, we compute what a random predictor would achieve.
With ~110 m/z bins and a typical horizon containing N unique bins, the random
hit rate for top-k is approximately `1 - (1 - N/110)^k`.


In [ ]:
# @title Cell 9: Compute random baseline hit rates
print("Random baseline (expected hit rates):")
print(f"{'Horizon':>10}  {'Top-k':>6}  {'Random Hit':>12}  {'Model Hit (LSTM)':>18}")
print("-" * 55)

for h in HORIZONS:
    # Compute average number of unique m/z bins in this horizon across test samples
    unique_bins_counts = []
    for sample in test_samples:
        rt_sec = sample["rt_seconds"]
        for pos in range(CONTEXT_LENGTH, len(sample["mz_bin"])):
            current_rt = rt_sec[pos - 1]
            future_mask = (rt_sec[pos:] > current_rt) & (rt_sec[pos:] <= current_rt + h)
            if future_mask.any():
                unique_bins_counts.append(len(set(sample["mz_bin"][pos:][future_mask])))
    avg_bins = np.mean(unique_bins_counts) if unique_bins_counts else 0

    for k in TOP_K_VALUES:
        # Random hit probability: 1 - P(none of k guesses match any of avg_bins targets)
        random_hit = 1 - ((1 - avg_bins / MAX_MZ_BIN) ** k) if MAX_MZ_BIN > 0 else 0
        lstm_hit = lstm_results[h][k]["hits"] / lstm_results[h][k]["total"] if lstm_results[h][k]["total"] > 0 else 0
        print(f"{h:>8}s  {k:>6}  {random_hit:>12.1%}  {lstm_hit:>18.1%}")

    if h < HORIZONS[-1]:
        print()


## 6. Figures


In [ ]:
# @title Cell 10: Generate 4-panel figure
fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# --- Panel A: Hit rate vs horizon (both models) ---
ax = axes[0, 0]
x = np.arange(len(HORIZONS))
width = 0.15
for i, k in enumerate(TOP_K_VALUES):
    lstm_hits = [lstm_results[h][k]["hits"] / lstm_results[h][k]["total"]
                 if lstm_results[h][k]["total"] > 0 else 0 for h in HORIZONS]
    tfm_hits = [tfm_results[h][k]["hits"] / tfm_results[h][k]["total"]
                if tfm_results[h][k]["total"] > 0 else 0 for h in HORIZONS]
    ax.bar(x - width*2 + i*width, lstm_hits, width, label=f'LSTM top-{k}',
           alpha=0.8, edgecolor='white')
    ax.bar(x + i*width, tfm_hits, width, label=f'TFM top-{k}',
           alpha=0.5, edgecolor='white', hatch='//')
ax.set_xticks(x)
ax.set_xticklabels([f'{h}s' for h in HORIZONS])
ax.set_xlabel('Prediction Horizon', fontsize=11)
ax.set_ylabel('Hit Rate', fontsize=11)
ax.set_title('A. Hit Rate by Horizon and Top-k', fontsize=12, fontweight='bold')
ax.legend(fontsize=7, ncol=2, loc='lower right')
ax.set_ylim(0, 1.05)

# --- Panel B: Coverage vs horizon ---
ax = axes[0, 1]
for i, k in enumerate(TOP_K_VALUES):
    lstm_cov = [lstm_results[h][k]["coverage_num"] / lstm_results[h][k]["coverage_den"]
                if lstm_results[h][k]["coverage_den"] > 0 else 0 for h in HORIZONS]
    ax.plot(HORIZONS, lstm_cov, 'o-', label=f'LSTM top-{k}', markersize=6)
ax.set_xlabel('Prediction Horizon (seconds)', fontsize=11)
ax.set_ylabel('Coverage (fraction of future features predicted)', fontsize=11)
ax.set_title('B. Feature Coverage by Horizon', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.set_ylim(0, 1.05)
ax.axhline(1.0, color='gray', linestyle='--', alpha=0.3)

# --- Panel C: Accuracy by RT position (LSTM) ---
ax = axes[1, 0]
pos_df = lstm_pos_data.copy()
pos_df['rt_bin_10pct'] = (pos_df['rt_fraction'] * 10).astype(int).clip(0, 9)
bin_acc = pos_df.groupby('rt_bin_10pct')['correct'].agg(['mean', 'count'])
bar_labels = [f'{i*10}-{(i+1)*10}%' for i in range(10)]
colors = plt.cm.viridis(np.linspace(0.2, 0.8, 10))
ax.bar(range(len(bin_acc)), bin_acc['mean'], color=colors[:len(bin_acc)],
       edgecolor='white', linewidth=0.5)
ax.set_xticks(range(len(bin_acc)))
ax.set_xticklabels(bar_labels[:len(bin_acc)], rotation=45, ha='right', fontsize=8)
ax.set_xlabel('Position in Run (RT percentile)', fontsize=11)
ax.set_ylabel('Top-1 Accuracy', fontsize=11)
ax.set_title('C. Prediction Accuracy by Run Position (LSTM)', fontsize=12, fontweight='bold')
ax.set_ylim(0, 1.05)
# Add count labels
for i, (_, row) in enumerate(bin_acc.iterrows()):
    ax.text(i, row['mean'] + 0.02, f'n={int(row["count"]):,}',
            ha='center', va='bottom', fontsize=6)

# --- Panel D: Hit rate improvement over random ---
ax = axes[1, 1]
# Compute random baselines for each horizon
random_baselines = {}
for h in HORIZONS:
    unique_bins_counts = []
    for sample in test_samples:
        rt_sec = sample["rt_seconds"]
        for pos in range(CONTEXT_LENGTH, len(sample["mz_bin"])):
            current_rt = rt_sec[pos - 1]
            future_mask = (rt_sec[pos:] > current_rt) & (rt_sec[pos:] <= current_rt + h)
            if future_mask.any():
                unique_bins_counts.append(len(set(sample["mz_bin"][pos:][future_mask])))
    avg_bins = np.mean(unique_bins_counts) if unique_bins_counts else 0
    random_baselines[h] = avg_bins

k_plot = 5  # Use top-5 for this panel
improvements = []
random_rates = []
model_rates = []
for h in HORIZONS:
    random_hit = 1 - ((1 - random_baselines[h] / MAX_MZ_BIN) ** k_plot)
    model_hit = lstm_results[h][k_plot]["hits"] / lstm_results[h][k_plot]["total"] if lstm_results[h][k_plot]["total"] > 0 else 0
    random_rates.append(random_hit)
    model_rates.append(model_hit)
    improvements.append(model_hit / random_hit if random_hit > 0 else 0)

x = np.arange(len(HORIZONS))
ax.bar(x - 0.15, random_rates, 0.3, label='Random', color='lightcoral', edgecolor='white')
ax.bar(x + 0.15, model_rates, 0.3, label='LSTM (top-5)', color='steelblue', edgecolor='white')
ax.set_xticks(x)
ax.set_xticklabels([f'{h}s' for h in HORIZONS])
ax.set_xlabel('Prediction Horizon', fontsize=11)
ax.set_ylabel('Hit Rate', fontsize=11)
ax.set_title('D. Model vs Random Baseline (Top-5)', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim(0, 1.05)
# Annotate fold improvement
for i, imp in enumerate(improvements):
    ax.text(i, max(random_rates[i], model_rates[i]) + 0.03,
            f'{imp:.1f}x', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()

fig_path = f"{OUTPUT_DIR}/acquisition_simulation.png"
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
print(f"Figure saved: {fig_path}")
plt.show()


## 7. Save Results


In [ ]:
# @title Cell 11: Save results JSON
import json

def results_to_dict(results, name):
    out = {}
    for h in HORIZONS:
        out[f"{h}s"] = {}
        for k in TOP_K_VALUES:
            r = results[h][k]
            hit_rate = r["hits"] / r["total"] if r["total"] > 0 else 0
            coverage = r["coverage_num"] / r["coverage_den"] if r["coverage_den"] > 0 else 0
            out[f"{h}s"][f"top{k}"] = {
                "hit_rate": round(hit_rate, 4),
                "coverage": round(coverage, 4),
                "n_predictions": r["total"],
            }
    return out

summary = {
    "experiment": "Acquisition Simulation (Phase 5)",
    "description": "Simulated predictive acquisition at 5/15/30s horizons",
    "models": {
        "LSTM": {
            "params": count_parameters(lstm_model),
            "best_epoch": lstm_ckpt["best_epoch"],
            "results": results_to_dict(lstm_results, "LSTM"),
        },
        "Transformer": {
            "params": count_parameters(tfm_model),
            "best_epoch": tfm_ckpt["best_epoch"],
            "results": results_to_dict(tfm_results, "Transformer"),
        },
    },
    "configuration": {
        "horizons_seconds": HORIZONS,
        "top_k_values": TOP_K_VALUES,
        "context_length": CONTEXT_LENGTH,
        "mz_bin_width_da": MZ_BIN_WIDTH,
        "num_mz_bins": MAX_MZ_BIN,
        "n_test_samples": len(test_samples),
    },
}

json_path = f"{OUTPUT_DIR}/acquisition_simulation_results.json"
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=2)
print(f"Results saved: {json_path}")

# Save position-level data
pos_path = f"{OUTPUT_DIR}/lstm_position_accuracy.csv"
lstm_pos_data.to_csv(pos_path, index=False)
print(f"Position data saved: {pos_path}")

# Print key takeaway
print()
print("=" * 60)
print("KEY RESULTS")
print("=" * 60)
for name, results in [("LSTM", lstm_results), ("Transformer", tfm_results)]:
    print(f"\n{name}:")
    for h in HORIZONS:
        r5 = results[h][5]
        hit = r5["hits"] / r5["total"] if r5["total"] > 0 else 0
        print(f"  {h:>2}s horizon, top-5: {hit:.1%} hit rate ({r5['total']:,} predictions)")


## Done!

**Outputs saved to `outputs/`:**
- `acquisition_simulation.png` — 4-panel figure (hit rates, coverage, position accuracy, vs random)
- `acquisition_simulation_results.json` — full results for all horizons and top-k values
- `lstm_position_accuracy.csv` — per-position prediction data

**Key question answered:** Does the model's 98% next-token accuracy translate to
practical utility for predictive acquisition? The hit rate at each horizon tells us
how reliably the instrument could pre-schedule MS/MS scans using model predictions.
